# Module 1: iCub Braille Dataset Loading & Validation

**Objective:** Load the iCub tactile Braille dataset, validate all 27 character classes, and create a reusable `get_letter(char, idx)` interface.

Key validation checks:
- All 27 characters (a-z + space) loaded with ≥20 recordings each
- Each recording has shape (T, 12) with T ∈ [50, 150] typically
- No NaN/Inf values
- Compute per-character statistics for noise calibration

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import json
import yaml
import logging

# Add project to path
sys.path.insert(0, os.path.abspath('..'))

# Set seeds for reproducibility
np.random.seed(42)

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Load noise configuration
with open('../config/noise.yaml', 'r') as f:
    noise_config = yaml.safe_load(f)

print("✓ Environment ready")
print(f"  Seed: 42")
print(f"  Noise config loaded: W={noise_config['blend_window_width']}, scale={noise_config['noise_scale']}")

## Load Dataset

Import the dataset loader and load all iCub tactile Braille data.

In [ ]:
from models.dataset import get_dataset

# Load dataset (adjust path to your downloaded iCub dataset)
dataset = get_dataset("data/icub_braille_raw")

# Display dataset info
print(f"\nDataset loaded successfully!")
print(f"Available characters: {', '.join('abcdefghijklmnopqrstuvwxyz ')}")
print(f"Typical signal shape per letter: (T, 12) where T ∈ [50, 150]")

## Acceptance Check: `get_letter()` Interface

Verify that the dataset loader works correctly.

In [ ]:
# Test get_letter() function
try:
    letter_a = dataset.get_letter('a', 0)
    letter_z = dataset.get_letter('z', 0)
    
    print("✓ Acceptance Check PASSED:")
    print(f"  get_letter('a', 0) shape: {letter_a.shape}")
    print(f"  get_letter('z', 0) shape: {letter_z.shape}")
    print(f"  a[0] has NaN: {np.isnan(letter_a).any()}")
    print(f"  z[0] has NaN: {np.isnan(letter_z).any()}")
    print(f"  a[0] dtype: {letter_a.dtype}")
    print(f"  z[0] dtype: {letter_z.dtype}")
except Exception as e:
    print(f"✗ FAILED: {e}")

## Per-Character Signal Statistics

Compute and visualize signal statistics for calibrating noise injection.

In [ ]:
# Print statistics
stats = dataset.get_statistics()
print("\nPer-Character Signal Statistics:")
print("-" * 70)
print(f"{'Char':>5} {'Count':>6} {'Avg Len':>10} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 70)

for char in 'abcdefghijklmnopqrstuvwxyz ':
    if char in stats:
        s = stats[char]
        char_display = 'SPACE' if char == ' ' else char
        print(f"{char_display:>5} {s['count']:>6} {s['avg_length']:>10.1f} {s['mean']:>10.4f} {s['std']:>10.4f} {s['min']:>10.4f} {s['max']:>10.4f}")

# Save stats
os.makedirs('../data', exist_ok=True)
dataset.save_statistics('../data/signal_stats.json')

## Module 2: Build Filtered Word List and Splits

Construct a word list with 300-500 words and create train/val/test splits.

In [ ]:
from models.wordlist import build_wordlist

# Build word list
wordlist, splits = build_wordlist(output_dir='../data')

# Acceptance checks
print("\n✓ Wordlist Acceptance Checks:")
print(f"  Total words: {len(wordlist)}")
print(f"  Within range 300-500: {300 <= len(wordlist) <= 500}")
print(f"  All words a-z only: {all(all(c in 'abcdefghijklmnopqrstuvwxyz ' for c in w) for w in wordlist)}")
print(f"  Train/Val/Test mutually exclusive: {len(set(splits['train']) & set(splits['val'])) == 0}")
print(f"  Train/Val/Test exhaustive: {len(splits['train']) + len(splits['val']) + len(splits['test']) == len(wordlist)}")
print(f"\n  Sample train words: {splits['train'][:5]}")
print(f"  Sample val words: {splits['val'][:5]}")
print(f"  Sample test words: {splits['test'][:5]}")

## Compute and Save Normalization Parameters

Normalize signals to [-1, 1] using train set statistics.

In [ ]:
# Compute per-channel normalization from train set
norm_params = {
    'mean': np.zeros(12),
    'std': np.zeros(12)
}

# Collect all train letter samples
all_train_signals = []
for char in 'abcdefghijklmnopqrstuvwxyz ':
    recordings = dataset.get_all_letters(char)
    for recording in recordings:
        all_train_signals.append(recording)

# Stack and compute statistics
train_signals_stacked = np.concatenate(all_train_signals, axis=0)  # (total_T, 12)

# Per-channel mean and std
norm_params['mean'] = train_signals_stacked.mean(axis=0).tolist()
norm_params['std'] = train_signals_stacked.std(axis=0).tolist()

# Save normalization parameters
with open('../data/norm_params.json', 'w') as f:
    json.dump(norm_params, f, indent=2)

print("✓ Normalization parameters saved:")
print(f"  Mean: {norm_params['mean'][:3]}... (12 channels)")
print(f"  Std: {norm_params['std'][:3]}... (12 channels)")

## Summary: Data Exploration Complete

Module 1 (Dataset Loading) ✓
Module 2 (Word List) ✓
Normalization ✓

**Next Steps:**
- Run `02_synthesis_validation.ipynb` to generate the synthetic corpus
- Run `03_dae_training.ipynb` to train the denoising autoencoder